# TT indicators for the 3 new mule accounts (v2)

**Purpose:** find inward-TT red flags that would have identified the 3 accounts business flagged, in a form another
monitoring system can use. Standalone and exploratory — not part of the behavioural screen.

**What v1 taught us:** the money arrives by inward TT (`TRAN_RMKS` starting **IT**) but leaves by ordinary debits — mostly
remarks starting **PA** — not by outward TT. So in v2, *money out* = every customer debit except fees and taxes, and the
rules focus on how the inward TTs arrive: who sends them, how many senders, how they are split, how young the account is.

**Steps**
1. Load: the mule accounts' full history; all IT/OT TTs on USD accounts since `TT_FROM` (cached from v1); and the debits of
   every account that received an inward TT (new query, cached).
2. The 3 accounts as they are: what came in, what went out, to whom.
3. Features per account and where the 3 sit against every other account that received an inward TT.
4. Candidate rules across thresholds: mules caught vs alerts per month.
5. Senders shared between mules, and who else they paid.

In [5]:
import os, re
from pathlib import Path
import numpy as np
import pandas as pd

SYNTHETIC = False

NEW_MULES = ["UG6302468", "UG6300154", "UG6334629"]  # the 3 accounts (ACID or account number)
INCLUDE_WATCHLIST = True                              # original mules: check rules on accounts not used to find them
TT_FROM = "2025-01-01"                                # book-wide window for comparison and alert volumes

# Debits that are fees, taxes or charges rather than money moving on (by remark prefix; check section 2's breakdown)
OUT_EXCLUDE_PREFIXES = ("FE", "EX", "AC", "TA")
DEBIT_MAX_AGE_DAYS = 7                                # refresh the cached debits after this many days

AS_OF = pd.Timestamp.today().normalize()
OUT = Path("outputs/tt_indicators"); OUT.mkdir(parents=True, exist_ok=True)
CACHE_DIR = Path("cache/tt_indicators" + ("_synthetic" if SYNTHETIC else "")); CACHE_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 250, "display.max_columns", 40, "display.max_colwidth", 70)

## 1. Connections and queries

In [6]:
ENV_FILE = ".env"
try:
    from dotenv import load_dotenv
    load_dotenv(ENV_FILE, override=False)
except ImportError:
    print("python-dotenv not installed; relying on variables already in the environment")

def _engine(prefix):
    '''Copied from aml_screening.py. READ = Finacle, WRITE = reporting DB (SELECTs only here).'''
    from sqlalchemy import create_engine, URL
    full_url = os.environ.get(f"AML_{prefix}_URL")
    if full_url:
        return create_engine(full_url, fast_executemany=True)
    def req(name):
        val = os.environ.get(f"AML_{prefix}_{name}")
        if not val:
            raise RuntimeError(f"Set AML_{prefix}_{name} (or provide a full AML_{prefix}_URL)")
        return val
    url = URL.create("mssql+pyodbc", username=req("USER"), password=req("PWD"), host=req("HOST"),
                     port=int(os.environ.get(f"AML_{prefix}_PORT", "1433")), database=req("DB"),
                     query={"driver": os.environ.get("AML_ODBC_DRIVER", "ODBC Driver 17 for SQL Server")})
    return create_engine(url, fast_executemany=True)

_fin = _rep = None
def fin():
    global _fin, _rep
    if _fin is None:
        _fin, _rep = _engine("READ"), _engine("WRITE")
    return _fin
def rep():
    fin(); return _rep

HTD = "RSVRRealtimeFinacleUGPRD.TBAADM.HIST_TRAN_DTL_TABLE"
GAM = "RSVRRealtimeFinacleUGPRD.TBAADM.GENERAL_ACCT_MAST_TABLE"
COLS = '''hd.ACID, gm.FORACID, gm.ACCT_NAME, gm.ACCT_OPN_DATE, hd.TRAN_ID, hd.TRAN_DATE, hd.RCRE_TIME, hd.TRAN_AMT,
          hd.PART_TRAN_TYPE, hd.TRAN_RMKS, hd.TRAN_PARTICULAR, hd.RPT_CODE'''

MONTH_TT_SQL = f'''
SELECT {COLS}
FROM {HTD} hd JOIN {GAM} gm ON hd.ACID = gm.ACID
WHERE gm.ACCT_CRNCY_CODE = 'USD'
  AND hd.TRAN_DATE >= '{{start}}' AND hd.TRAN_DATE < '{{end}}'
  AND (hd.TRAN_RMKS LIKE 'IT%' OR hd.TRAN_RMKS LIKE 'OT%')
'''
MULE_TXN_SQL = f'''
SELECT {COLS}
FROM {HTD} hd JOIN {GAM} gm ON hd.ACID = gm.ACID
WHERE hd.ACID IN ({{ids}}) OR gm.FORACID IN ({{ids}})
'''
DEBITS_SQL = f'''
SELECT {COLS}
FROM {HTD} hd JOIN {GAM} gm ON hd.ACID = gm.ACID
WHERE hd.ACID IN ({{ids}}) AND hd.TRAN_DATE >= '{{start}}' AND hd.PART_TRAN_TYPE = 'D'
'''

def _in_list(ids):
    ids = [str(i).strip() for i in ids]
    bad = [i for i in ids if not re.fullmatch(r"[A-Za-z0-9_\-]+", i)]
    if bad:
        raise ValueError(f"Unexpected identifier format: {bad[:5]}")
    return ",".join(f"'{i}'" for i in ids)

def read_chunks(template, ids, eng, chunk=500, **kw):
    ids = list(dict.fromkeys(map(str, ids)))
    parts = [pd.read_sql(template.format(ids=_in_list(ids[i:i + chunk]), **kw), eng) for i in range(0, len(ids), chunk)]
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

## 2. Synthetic data (dry-run only)

In [7]:
def make_synthetic(seed=21):
    '''Fake USD accounts. Mules are funded by inward TTs (one shared sender who splits amounts; one mule with many small
    senders) and paid out by PA debits within hours. Ordinary accounts: importers (OT), exporters (IT, slow PA), and
    legitimate fast intermediaries. Fees are FE debits. Not evidence of anything.'''
    rng = np.random.default_rng(seed)
    rows, n = [], [0]
    names = [f"PARTY {i:04d} LTD" for i in range(600)]
    people = [f"PERSON {i:04d}" for i in range(400)]
    def tx(acid, fora, name, opn, when, amt, ptt, rmk, party):
        n[0] += 1
        rows.append(dict(ACID=acid, FORACID=fora, ACCT_NAME=name, ACCT_OPN_DATE=opn, TRAN_ID=f"T{n[0]}",
                         TRAN_DATE=when.normalize(), RCRE_TIME=when, TRAN_AMT=round(float(amt), 2), PART_TRAN_TYPE=ptt,
                         TRAN_RMKS=f"{rmk}{rng.integers(10**6, 10**7)}", TRAN_PARTICULAR=party, RPT_CODE=None))
    def pay_out(a, f, nm, o, when, amt):
        tx(a, f, nm, o, when + pd.Timedelta(hours=float(rng.uniform(1, 20))), amt * rng.uniform(.97, .999), "D", "PA",
           f"PAYMENT TO {rng.choice(names)}")
        tx(a, f, nm, o, when + pd.Timedelta(hours=21), 15, "D", "FE", "TRANSFER FEE")
    for i in range(1500):
        a, f = f"UG{i:07d}", f"9{i:09d}"
        o = pd.Timestamp("2023-06-01") + pd.Timedelta(days=int(rng.integers(0, 1200)))
        start = max(o, pd.Timestamp("2024-06-01"))
        if i < 13:
            nm = f"SYNTHETIC MULE {i:02d}"; t0 = o + pd.Timedelta(days=int(rng.integers(2, 40)), hours=8)
            if i in (0, 2, 5):                                        # the shared sender, splitting into identical TTs
                amt = round(rng.uniform(100_000, 600_000), -1)
                for k in range(int(rng.integers(2, 5))):
                    tx(a, f, nm, o, t0, amt, "C", "IT", "1/Example Sender Name")
                    pay_out(a, f, nm, o, t0, amt)
            senders = rng.choice(people, 30 if i == 1 else 4, replace=False)
            for k in range(int(rng.integers(6, 40)) if i == 1 else int(rng.integers(3, 8))):
                when = t0 + pd.Timedelta(days=int(rng.integers(0, 25)), minutes=int(rng.integers(0, 600)))
                amt = rng.uniform(3_000, 60_000) if i == 1 else rng.uniform(50_000, 400_000)
                tx(a, f, nm, o, when, amt, "C", "IT", f"1/{rng.choice(senders)}")
                pay_out(a, f, nm, o, when, amt)
        else:
            nm = f"SYNTHETIC CUSTOMER {i:05d}"; r = rng.random()
            for m in range(int(rng.integers(2, 18))):
                when = start + pd.Timedelta(days=int(rng.integers(0, max((AS_OF - start).days, 2))), hours=float(rng.uniform(8, 17)))
                if r < .40:      # importer
                    tx(a, f, nm, o, when, rng.uniform(5_000, 150_000), "D", "OT", f"{rng.choice(names)}")
                elif r < .85:    # exporter: receives, pays out slowly
                    amt = rng.uniform(5_000, 200_000)
                    tx(a, f, nm, o, when, amt, "C", "IT", f"{rng.choice(names)}")
                    tx(a, f, nm, o, when + pd.Timedelta(days=int(rng.integers(5, 40))), amt * .5, "D", "PA", f"PAYMENT TO {rng.choice(names)}")
                else:            # legitimate fast intermediary
                    amt = rng.uniform(20_000, 300_000)
                    tx(a, f, nm, o, when, amt, "C", "IT", f"{rng.choice(names)}")
                    pay_out(a, f, nm, o, when, amt)
    df = pd.DataFrame(rows)
    df = df[df.RCRE_TIME <= AS_OF]
    return df, [f"UG{i:07d}" for i in range(3)], [f"UG{i:07d}" for i in range(3, 13)]

## 3. Load — cache first (TTs and mule history are reused from v1)

In [8]:
if SYNTHETIC:
    syn_all, syn_new, syn_watch = make_synthetic()
    NEW_MULES = syn_new
watch = (syn_watch if SYNTHETIC else
         (pd.read_sql("SELECT acid FROM aml_watchlist WHERE active = 1", rep())["acid"].tolist() if INCLUDE_WATCHLIST else []))
mule_ids = list(dict.fromkeys([str(x) for x in NEW_MULES] + [str(x) for x in watch]))
queries = []

p = CACHE_DIR / f"mule_txns_{len(mule_ids)}.pkl"
if p.exists():
    mule_tx = pd.read_pickle(p)
else:
    mule_tx = (syn_all[syn_all.ACID.isin(mule_ids) | syn_all.FORACID.isin(mule_ids)].copy() if SYNTHETIC
               else pd.read_sql(MULE_TXN_SQL.format(ids=_in_list(mule_ids)), fin()))
    mule_tx.to_pickle(p); queries.append(f"mule accounts: {len(mule_tx):,} transactions")

parts = []
for m in pd.period_range(pd.Timestamp(TT_FROM).to_period("M"), AS_OF.to_period("M"), freq="M"):
    f = CACHE_DIR / f"tt_{m}.pkl"
    if not f.exists() or m == AS_OF.to_period("M"):
        s, e = m.start_time, (m + 1).start_time
        t = (syn_all[(syn_all.TRAN_DATE >= s) & (syn_all.TRAN_DATE < e) & syn_all.TRAN_RMKS.str[:2].isin(["IT", "OT"])].copy()
             if SYNTHETIC else pd.read_sql(MONTH_TT_SQL.format(start=s.strftime("%Y-%m-%d"), end=e.strftime("%Y-%m-%d")), fin()))
        t.to_pickle(f); queries.append(f"TTs {m}: {len(t):,}")
    parts.append(pd.read_pickle(f))
book_tt = pd.concat(parts, ignore_index=True)

# debits of every non-mule account that received an inward TT (to see where the money went)
mule_acids = set(mule_tx.ACID)
it_accts = sorted(set(book_tt.loc[book_tt.TRAN_RMKS.fillna("").str.upper().str.startswith("IT"), "ACID"]) - mule_acids)
pd_ = CACHE_DIR / f"debits_{TT_FROM}.pkl"; pm = CACHE_DIR / f"debits_{TT_FROM}_meta.pkl"
debits = pd.read_pickle(pd_) if pd_.exists() else None
fetched = pd.read_pickle(pm).fetched_on.iloc[0] if pm.exists() else None
stale = debits is None or fetched is None or (AS_OF - fetched).days > DEBIT_MAX_AGE_DAYS
need = it_accts if stale else sorted(set(it_accts) - set(debits.ACID))
if need:
    new = (syn_all[syn_all.ACID.isin(need) & (syn_all.PART_TRAN_TYPE == "D") & (syn_all.TRAN_DATE >= TT_FROM)].copy() if SYNTHETIC
           else read_chunks(DEBITS_SQL, need, fin(), start=TT_FROM))
    debits = new if stale else pd.concat([debits, new], ignore_index=True)
    debits.to_pickle(pd_)
    if stale:
        pd.DataFrame({"fetched_on": [AS_OF]}).to_pickle(pm)
    queries.append(f"debits: {len(need):,} accounts, {len(new):,} rows")
print("Finacle queries this run:", "; ".join(queries) if queries else "none — everything served from cache")

def prep(df):
    df = df.copy()
    for col in ("TRAN_DATE", "RCRE_TIME", "ACCT_OPN_DATE"):
        df[col] = pd.to_datetime(df[col], errors="coerce")
    df["when"] = df.RCRE_TIME.fillna(df.TRAN_DATE)
    df["rmk"] = df.TRAN_RMKS.fillna("").str.strip().str.upper()
    df["prefix"] = df.rmk.str[:2]
    df["is_cr"] = df.PART_TRAN_TYPE.fillna("").str.upper().str.strip() == "C"
    df["is_charge"] = df.RPT_CODE.fillna("").str.upper().isin(["CHG", "INT"])
    df["TRAN_AMT"] = df.TRAN_AMT.astype(float).abs()
    df["flow"] = np.select([df.is_cr & (df.prefix == "IT"),
                            ~df.is_cr & ~df.is_charge & ~df.prefix.isin(OUT_EXCLUDE_PREFIXES)], ["in", "out"], "")
    df["party"] = (df.TRAN_PARTICULAR.fillna("").str.upper()
                     .str.replace(r"^\s*(IT|OT)\s*[/:-]?\s*", "", regex=True)
                     .str.replace(r"^\s*\d+\s*/\s*", "", regex=True)                       # SWIFT line numbers such as "1/"
                     .str.replace(r"\d{6,}", "#", regex=True).str.replace(r"\s+", " ", regex=True).str.strip())
    return df.drop_duplicates(["ACID", "TRAN_ID", "TRAN_DATE", "PART_TRAN_TYPE", "TRAN_AMT"])

mule_tx = prep(mule_tx)
others = prep(pd.concat([book_tt, debits], ignore_index=True))
others = others[~others.ACID.isin(mule_acids)]
newset = set(map(str, NEW_MULES))
ids_tbl = mule_tx[["ACID", "FORACID", "ACCT_NAME"]].drop_duplicates("ACID")
new_acids = ids_tbl[ids_tbl.ACID.astype(str).isin(newset) | ids_tbl.FORACID.astype(str).isin(newset)].ACID.tolist()
old_acids = [a for a in ids_tbl.ACID if a not in new_acids]
group = {**{a: "new mule" for a in new_acids}, **{a: "original mule" for a in old_acids}}
names = ids_tbl.set_index("ACID").ACCT_NAME
print(f"Mule accounts: {len(new_acids)} new, {len(old_acids)} original | other accounts with an inward TT since {TT_FROM}: {len(it_accts):,}")
print("\nDebit remark prefixes across those accounts (excluded from 'money out':", ", ".join(OUT_EXCLUDE_PREFIXES) + "):")
dd = others[~others.is_cr]
print(dd.groupby("prefix").TRAN_AMT.agg(["size", "sum", "median"]).sort_values("sum", ascending=False).head(12)
        .rename(columns={"size": "count", "sum": "$", "median": "median $"}).round(0).to_string())

Finacle queries this run: TTs 2026-10: 96,210; debits: 13,484 accounts, 4,495,450 rows
Mule accounts: 3 new, 10 original | other accounts with an inward TT since 2025-01-01: 13,484

Debit remark prefixes across those accounts (excluded from 'money out': FE, EX, AC, TA):
          count             $  median $
prefix                                 
        1449354  4.666817e+11      28.0
OT       710500  1.376673e+10      15.0
IT       348112  1.156494e+10     600.0
PA       206236  9.240614e+09    3932.0
TZ          518  3.054161e+09     300.0
AT        40427  3.042421e+09    7000.0
CT        12874  2.274842e+09      10.0
99       180284  8.668469e+08      10.0
UG         8596  8.308147e+08      36.0
07        70734  8.048421e+08    3000.0
GP         1805  7.323288e+08   48000.0
BO          493  5.694808e+08     203.0


## 4. The 3 accounts as they are: what came in, what went out, to whom

In [9]:
for a in new_acids:
    t = mule_tx[mule_tx.ACID == a]
    i, o = t[t.flow == "in"], t[t.flow == "out"]
    print("=" * 110)
    print(f"{names[a]}  |  opened {t.ACCT_OPN_DATE.min():%Y-%m-%d}  |  in by IT ${i.TRAN_AMT.sum():,.0f} ({len(i)} TTs, "
          f"{i.party.nunique()} senders)  |  out ${o.TRAN_AMT.sum():,.0f} ({len(o)} debits, {o.party.nunique()} receivers)")
    print("\nMoney out by remark prefix:")
    print(o.groupby("prefix").TRAN_AMT.agg(["size", "sum"]).rename(columns={"size": "count", "sum": "$"}).round(0).to_string())
    print("\nFirst 10 money-out debits as recorded:")
    print(o.sort_values("when")[["when", "TRAN_AMT", "TRAN_RMKS", "TRAN_PARTICULAR"]].head(10).to_string(index=False))
    print("\nLargest receivers:")
    print(o.groupby("party").TRAN_AMT.agg(["size", "sum"]).sort_values("sum", ascending=False).head(8)
            .rename(columns={"size": "count", "sum": "$"}).round(0).to_string())

TYR TECH -SMC LTD  |  opened 2026-02-26  |  in by IT $5,074,200 (62 TTs, 36 senders)  |  out $5,072,013 (71 debits, 23 receivers)

Money out by remark prefix:
        count          $
prefix                  
            7       18.0
PA         64  5071995.0

First 10 money-out debits as recorded:
               when  TRAN_AMT            TRAN_RMKS TRAN_PARTICULAR
2026-03-03 09:59:34    7492.0 PAYEX-86715459950265            DADA
2026-03-04 09:19:55   10000.0 PAYEX-86748002967479            DADA
2026-03-04 15:27:08   17000.0 PAYEX-86767232978006            DADA
2026-03-05 10:14:07    9884.0 PAYEX-86781833987817            DADA
2026-03-06 14:54:35   28000.0 PAYEX-86837462017709          FASBIR
2026-03-09 10:46:13   80000.0 PAYEX-86858050033793       Estifanos
2026-03-09 10:57:25   37131.0 PAYEX-86858632034001         MERHAWI
2026-03-10 10:08:16   35141.0 PAYEX-86884954048982         TAURARO
2026-03-10 16:02:42    2390.0 PAYEX-86905426060177          FASBIR
2026-03-11 09:51:47   33128.0 P

## 5. Features per account, and where the 3 sit

Mules: full history. Everyone else: since `TT_FROM`, accounts that received at least one inward TT. `hours to out` follows
each inward TT's money, first in first out, to the debits that took it out.

In [10]:
def clear_table(df):
    '''One row per inward TT: hours until money-out debits (first in, first out) had covered its full amount.'''
    parts = []
    for a, g in df[df.flow != ""].sort_values("when").groupby("ACID"):
        i, o = g[g.flow == "in"], g[g.flow == "out"]
        if i.empty:
            continue
        cum_out, t_out = o.TRAN_AMT.cumsum().to_numpy(), o.when.to_numpy()
        k = np.searchsorted(cum_out, i.TRAN_AMT.cumsum().to_numpy() - 1e-6)
        hrs = np.full(len(i), np.nan); ok = k < len(cum_out)
        hrs[ok] = (t_out[k[ok]] - i.when.to_numpy()[ok]) / np.timedelta64(1, "h")
        parts.append(i[["ACID", "when", "TRAN_AMT", "party", "TRAN_DATE"]].assign(hrs=np.clip(hrs, 0, None)))
    return pd.concat(parts, ignore_index=True)

def features(df, itc):
    rows, by = {}, {a: g for a, g in itc.groupby("ACID")}
    for a, g in df[df.flow != ""].sort_values("when").groupby("ACID"):
        i, o = g[g.flow == "in"], g[g.flow == "out"]
        if i.empty:
            continue
        c = by[a]; w, h = c.TRAN_AMT, c.hrs
        first = i.when.min()
        early = i[i.when < first + pd.Timedelta(days=30)]
        same = i.groupby(["party", "TRAN_DATE", "TRAN_AMT"]).size()
        rows[a] = {"IT count": len(i), "IT total $k": i.TRAN_AMT.sum() / 1e3,
                   "out total $k": o.TRAN_AMT.sum() / 1e3, "out / IT": o.TRAN_AMT.sum() / i.TRAN_AMT.sum(),
                   "out by OT share": o[o.prefix == "OT"].TRAN_AMT.sum() / max(o.TRAN_AMT.sum(), 1),
                   "IT money out within 24h": w[h <= 24].sum() / w.sum(), "IT money out within 72h": w[h <= 72].sum() / w.sum(),
                   "median hours to out": h.median(),
                   "distinct senders": i.party.nunique(), "senders in first 30 days": early.party.nunique(),
                   "distinct receivers": o.party.nunique(),
                   "largest IT $k": i.TRAN_AMT.max() / 1e3,
                   "max IT in 7 days $k": i.set_index("when").TRAN_AMT.rolling("7D").sum().max() / 1e3,
                   "identical ITs, same sender, same day (max)": int(same.max()),
                   "round-thousand ITs": (i.TRAN_AMT % 1000 == 0).mean(),
                   "days open at first IT": (first - g.ACCT_OPN_DATE.min()).days}
    return pd.DataFrame.from_dict(rows, orient="index")

itc = clear_table(pd.concat([mule_tx, others], ignore_index=True))
mule_feat = features(mule_tx, itc)
book_feat = features(others, itc)

def pct(col, v):
    s = book_feat[col].dropna()
    return np.nan if pd.isna(v) or s.empty else round(100 * (s <= v).mean())

shown = [a for a in new_acids if a in mule_feat.index]
view = mule_feat.loc[shown].T
view.columns = [str(names[a])[:22] for a in shown]
for a, col in zip(shown, list(view.columns)):
    view[col + " pctile"] = [pct(c, mule_feat.loc[a, c]) for c in view.index]
view["original mules (median)"] = mule_feat.loc[[a for a in old_acids if a in mule_feat.index]].median()
view["others with IT (median)"] = book_feat.median()
view["others p95"] = book_feat.quantile(.95)
print(f"Compared with {len(book_feat):,} other USD accounts that received an inward TT since {TT_FROM}. "
      "Percentile = share of those accounts at or below the mule's value.")
display(view.round(2))

Compared with 13,236 other USD accounts that received an inward TT since 2025-01-01. Percentile = share of those accounts at or below the mule's value.


,TYR TECH -SMC LTD,THRIFTOPIA UGANDA LIMI,QUENSPIRE UGANDA LIMIT,TYR TECH -SMC LTD pctile,THRIFTOPIA UGANDA LIMI pctile,QUENSPIRE UGANDA LIMIT pctile,original mules (median),others with IT (median),others p95
IT count,62.00,4.00,5.00,96,48,53,22.00,5.00,50.00
IT total $k,5074.20,2289.80,736.56,98,97,92,1983.21,19.83,1386.94
out total $k,5072.01,2289.61,736.32,95,92,86,4378.70,54.91,5545.68
out / IT,1.00,1.00,1.00,22,22,22,1.00,1.45,130.66
out by OT share,0.00,0.00,0.00,61,61,61,0.00,0.00,0.69
IT money out within 24h,0.80,0.75,0.64,59,57,54,0.55,0.50,1.00
IT money out within 72h,0.93,0.75,0.68,60,54,51,0.76,0.64,1.00
median hours to out,8.32,2.40,103.65,54,52,63,19.65,0.00,2960.27
distinct senders,36.00,1.00,4.00,99,49,85,17.00,2.00,11.00
senders in first 30 days,35.00,1.00,1.00,100,84,84,8.00,1.00,3.00


## 6. Candidate rules across thresholds

For each setting: which of the 3 it catches, how many original mules (not used to find these rules), and how many other
accounts it would have raised per month since `TT_FROM`. Prefer settings that also catch original mules.

In [11]:
months = max(len(pd.period_range(pd.Timestamp(TT_FROM).to_period("M"), AS_OF.to_period("M"))) - 1, 1)
F = pd.concat([mule_feat, book_feat])
kind = pd.Series({a: group.get(a, "other") for a in F.index})

def score(hit, label):
    hit = set(hit)
    n_other = sum(kind.get(a, "other") == "other" for a in hit)
    return {"rule": label, "new mules caught": ", ".join(str(names[a])[:18] for a in new_acids if a in hit) or "none",
            "of 3": sum(a in hit for a in new_acids), "original mules": f"{sum(a in hit for a in old_acids)} of {len(old_acids)}",
            "other accounts": n_other, "per month": round(n_other / months, 1)}

rows = []
for X in (100_000, 250_000, 500_000, 1_000_000):          # T1 young account, large inward TTs
    for D in (30, 90):
        rows.append(score(F[(F["days open at first IT"] <= D) & (F["max IT in 7 days $k"] * 1e3 >= X)].index,
                          f"T1 account <= {D} days old at first IT receives >= ${X/1e3:,.0f}k by IT within 7 days"))
for k in (2, 3):                                          # T2 split: identical TTs, same sender, same day
    for X in (0, 10_000, 50_000):
        same = itc[itc.TRAN_AMT >= X].groupby(["ACID", "party", "TRAN_DATE", "TRAN_AMT"]).size()
        rows.append(score(same[same >= k].index.get_level_values(0).unique(),
                          f"T2 >= {k} identical inward TTs from one sender on one day" + (f", each >= ${X/1e3:,.0f}k" if X else "")))
for N in (5, 10, 20):                                     # T3 many senders soon after the first inward TT
    for D in (90, None):
        m = F["senders in first 30 days"] >= N
        if D:
            m &= F["days open at first IT"] <= D
        rows.append(score(F[m].index, f"T3 >= {N} distinct senders in the 30 days after first IT" + (f", account <= {D} days old" if D else "")))
for X in (100_000, 250_000):                              # T4 pass-through via any debit
    for H in (24, 72):
        rows.append(score(itc[(itc.TRAN_AMT >= X) & (itc.hrs <= H)].ACID.unique(),
                          f"T4 inward TT >= ${X/1e3:,.0f}k paid out (any debit) within {H}h"))
# T5 receives from a sender who also funded a known mule (each mule judged against the OTHER mules' senders)
mule_send = itc[itc.ACID.isin(mule_acids)].groupby("party").ACID.apply(set)
t5 = set()
for a, g in itc.groupby("ACID"):
    for s in g.party.unique():
        if s in mule_send.index and (mule_send[s] - {a}):
            t5.add(a); break
rows.append(score(t5, "T5 receives an inward TT from a sender who also funded a known mule"))
# combinations
t1 = F[(F["days open at first IT"] <= 90) & (F["max IT in 7 days $k"] >= 500)].index
same = itc[itc.TRAN_AMT >= 50_000].groupby(["ACID", "party", "TRAN_DATE", "TRAN_AMT"]).size()
t2 = same[same >= 2].index.get_level_values(0).unique()
t3 = F[(F["senders in first 30 days"] >= 10) & (F["days open at first IT"] <= 90)].index
rows.append(score(set(t1) | set(t2) | set(t3) | t5, "ANY of: T1 (90d, $500k) / T2 (2+, $50k) / T3 (10, 90d) / T5"))
grid = pd.DataFrame(rows).set_index("rule")
display(grid)
print(f"'Other accounts' = distinct non-mule accounts over {months} complete months since {TT_FROM}. Mules are judged on full history.")

,new mules caught,of 3,original mules,other accounts,per month
rule,,,,,
T1 account <= 30 days old at first IT receives >= $100k by IT within 7 days,"TYR TECH -SMC LTD, QUENSPIRE UGANDA L",2,4 of 10,166,7.9
T1 account <= 90 days old at first IT receives >= $100k by IT within 7 days,"TYR TECH -SMC LTD, THRIFTOPIA UGANDA , QUENSPIRE UGANDA L",3,6 of 10,280,13.3
T1 account <= 30 days old at first IT receives >= $250k by IT within 7 days,"TYR TECH -SMC LTD, QUENSPIRE UGANDA L",2,4 of 10,101,4.8
T1 account <= 90 days old at first IT receives >= $250k by IT within 7 days,"TYR TECH -SMC LTD, THRIFTOPIA UGANDA , QUENSPIRE UGANDA L",3,6 of 10,167,8.0
T1 account <= 30 days old at first IT receives >= $500k by IT within 7 days,"TYR TECH -SMC LTD, QUENSPIRE UGANDA L",2,4 of 10,62,3.0
T1 account <= 90 days old at first IT receives >= $500k by IT within 7 days,"TYR TECH -SMC LTD, THRIFTOPIA UGANDA , QUENSPIRE UGANDA L",3,6 of 10,95,4.5
"T1 account <= 30 days old at first IT receives >= $1,000k by IT within 7 days",TYR TECH -SMC LTD,1,2 of 10,37,1.8
"T1 account <= 90 days old at first IT receives >= $1,000k by IT within 7 days","TYR TECH -SMC LTD, THRIFTOPIA UGANDA",2,3 of 10,51,2.4
T2 >= 2 identical inward TTs from one sender on one day,THRIFTOPIA UGANDA,1,2 of 10,481,22.9


'Other accounts' = distinct non-mule accounts over 21 complete months since 2025-01-01. Mules are judged on full history.


## 7. Senders shared between mules

Senders who funded two or more mule accounts, and how many other accounts they also sent money to. A sender who funds
only mules is the simplest list-based rule any system can run. Narration names are approximate — check them before use.

In [12]:
mi = itc[itc.ACID.isin(mule_acids) & (itc.party.str.len() > 3)]
shared = (mi.groupby("party").agg(mules=("ACID", "nunique"), to_mules_k=("TRAN_AMT", lambda s: round(s.sum() / 1e3)),
                                  first=("when", "min"), last=("when", "max"))
            .query("mules >= 2"))
oi = itc[~itc.ACID.isin(mule_acids)]
use = oi.groupby("party").agg(other_accounts=("ACID", "nunique"), to_others_k=("TRAN_AMT", lambda s: round(s.sum() / 1e3)))
shared = shared.join(use).fillna({"other_accounts": 0, "to_others_k": 0})
shared["mule names"] = [", ".join(sorted(str(names[a])[:18] for a in mi[mi.party == p].ACID.unique())) for p in shared.index]
shared = shared.sort_values(["mules", "other_accounts"], ascending=[False, True])
print(f"Senders that funded 2 or more mule accounts: {len(shared)}")
display(shared.head(30))

print("\nOther accounts funded by those senders (candidates for review, not conclusions):")
cand = oi[oi.party.isin(shared.index)].groupby("ACID").agg(senders=("party", lambda s: ", ".join(sorted(set(s)))[:60]),
                                                            received_k=("TRAN_AMT", lambda s: round(s.sum() / 1e3)))
display(cand.sort_values("received_k", ascending=False).head(25))

Senders that funded 2 or more mule accounts: 61


,mules,to_mules_k,first,last,other_accounts,to_others_k,mule names
party,,,,,,,
SANDY TRAN,4,163,2025-04-03 08:01:55,2025-05-22 08:02:08,4.0,10.0,"AETHERGRID TECHNOL, KWELIVEST INVESTME, NEBULACORE INNOVAT, PACIFI..."
THOMAS JURGEN SCHUNN,3,3339,2026-03-24 09:11:43,2026-04-15 08:11:38,0.0,0.0,"QUENSPIRE UGANDA L, THRIFTOPIA UGANDA , TYR TECH -SMC LTD"
BILLY NGO,3,43,2025-03-18 08:00:23,2025-04-22 08:03:16,1.0,0.0,"AETHERGRID TECHNOL, KWELIVEST INVESTME, NEBULACORE INNOVAT"
GARY BRADSHAW /OR GILMER GRAYLL BR,3,258,2025-03-20 08:01:50,2025-05-21 08:01:43,1.0,0.0,"AETHERGRID TECHNOL, KWELIVEST INVESTME, NEBULACORE INNOVAT"
BRET BISHOP,3,205,2025-04-02 08:01:46,2025-05-06 08:02:07,2.0,0.0,"AETHERGRID TECHNOL, NEBULACORE INNOVAT, PACIFIC PRINCE INT"
PHUONG M TRIEU,3,135,2025-03-21 09:36:50,2025-04-17 08:02:12,2.0,0.0,"AETHERGRID TECHNOL, NEBULACORE INNOVAT, PACIFIC PRINCE INT"
CHI LUU LLC DBA LIVING COLORS NAILS,3,78,2025-04-03 13:01:50,2025-04-30 12:09:12,4.0,5.0,"KWELIVEST INVESTME, NEBULACORE INNOVAT, PACIFIC PRINCE INT"
CHI V PHAM,3,53,2025-01-27 08:03:26,2025-04-23 08:02:44,4.0,10.0,"AETHERGRID TECHNOL, NEBULACORE INNOVAT, PACIFIC PRINCE INT"
DUYKIM TRAN,3,28,2025-04-07 08:02:36,2025-06-11 08:01:06,4.0,12.0,"KWELIVEST INVESTME, NEBULACORE INNOVAT, PACIFIC PRINCE INT"



Other accounts funded by those senders (candidates for review, not conclusions):


,senders,received_k
ACID,,
UG5856411,"JAMES NADEL, KHAM HOANG DO, KIM CHI THI NGUYEN, KPC INTERNAT",1403
UG6183885,"KPC INTERNATIONAL CORPORATION, SAT JAGROOP RANDHAWA",1228
UG6160539,"DUYKIM TRAN, JIMMY HOANG, JOHN LASPINA, KPC INTERNATIONAL CO",981
UG6247327,KPC INTERNATIONAL CORPORATION,918
UG6160591,THANG PHAM,775
UG6270030,"KPC INTERNATIONAL CORPORATION, SAT JAGROOP RANDHAWA",636
UG6212631,"KPC INTERNATIONAL CORPORATION, LOAN L BUI, SAT JAGROOP RANDH",633
UG6270217,KPC INTERNATIONAL CORPORATION,510
UG6103353,"JAMES NADEL, KHAM HOANG DO, KIM CHI THI NGUYEN, LIEN N PHAM,",212


## 8. Export

In [13]:
view.to_csv(OUT / "tt_profile_new_mules_v2.csv")
grid.to_csv(OUT / "tt_rule_grid_v2.csv")
shared.to_csv(OUT / "tt_shared_senders.csv")
cand.to_csv(OUT / "tt_accounts_funded_by_shared_senders.csv")
mule_feat.assign(group=mule_feat.index.map(group)).to_csv(OUT / "tt_features_mules_v2.csv")
print("Saved to", OUT.resolve())

Saved to /home/A270827/experiments/AML/outputs/tt_indicators


In [14]:
# Run inside tt_indicators.ipynb after all its sections:   %run -i tt_candidates_and_ruleset.py
# (A) REAL-TIME REPLAY of the candidate rules, as a near-real-time system would run them: transactions are replayed in
#     time order and each rule fires on an inward TT using only what was known at that moment.
# (B) The review list of accounts paid by senders who also funded mules, enriched.
# No new transaction pulls (one small account-master query for the review list).

T1_AGE_DAYS, T1_USD, T1_WINDOW = 90, 500_000, "7D"        # young account, large inflow by inward TT
T3_AGE_DAYS, T3_SENDERS, T3_WINDOW_DAYS = 90, 20, 30      # young account, many distinct senders
T2_COUNT, T2_MIN_USD = 3, 10_000                          # identical inward TTs, same sender, same day
BROAD_SENDER_MIN = 10                                     # a mule-linked sender who paid > this many other accounts is "broad"
COOLDOWN_DAYS = 30                                        # repeat alerts of the same rule on one account within this are one alert
HOLD_DAYS = 7                                             # outgoing payments within this many days after an alert go to review

broad = set(shared[shared.other_accounts > BROAD_SENDER_MIN].index)
ev = pd.concat([mule_tx[mule_tx.flow != ""], others[others.flow != ""]], ignore_index=True)
ev = ev[["ACID", "when", "TRAN_DATE", "TRAN_AMT", "flow", "party", "ACCT_OPN_DATE"]].dropna(subset=["when"])
ev = ev.sort_values(["ACID", "when"]).reset_index(drop=True)
mule_set = set(mule_acids)

# when did each sender first fund each mule (for "known sender": funded ANOTHER mule BEFORE this moment)
mi_ = ev[ev.ACID.isin(mule_set) & (ev.flow == "in") & (ev.party.str.len() > 3)]
first_fund = mi_.groupby(["party", "ACID"]).when.min()
fund_by_party = {p: g.droplevel(0).sort_values() for p, g in first_fund.groupby(level=0)}

def known_before(party, acct, t, exclude_broad):
    s = fund_by_party.get(party)
    if s is None or (exclude_broad and party in broad):
        return False
    s = s.drop(acct, errors="ignore")
    return bool(len(s)) and s.iloc[0] < t

# ---------- replay every inward TT ----------
fires = []
d7, d30 = pd.Timedelta(T1_WINDOW), pd.Timedelta(days=T3_WINDOW_DAYS)
for a, g in ev[ev.flow == "in"].groupby("ACID", sort=False):
    w = g.when.to_numpy(); amt = g.TRAN_AMT.to_numpy(); par = g.party.to_numpy()
    age = (g.when - g.ACCT_OPN_DATE).dt.days.to_numpy()
    same = (g.assign(big=g.TRAN_AMT >= T2_MIN_USD).groupby([g.TRAN_DATE, g.party, g.TRAN_AMT]).cumcount() + 1).to_numpy()
    lo7 = lo30 = 0; s7 = 0.0; cnt = {}
    for i in range(len(g)):
        s7 += amt[i]
        while w[lo7] <= w[i] - d7:
            s7 -= amt[lo7]; lo7 += 1
        if par[i]:
            cnt[par[i]] = cnt.get(par[i], 0) + 1
        while w[lo30] <= w[i] - d30:
            if par[lo30]:
                cnt[par[lo30]] -= 1
                if not cnt[par[lo30]]:
                    del cnt[par[lo30]]
            lo30 += 1
        hit = {"T1": age[i] <= T1_AGE_DAYS and s7 >= T1_USD,
               "T3": age[i] <= T3_AGE_DAYS and len(cnt) >= T3_SENDERS,
               "T2": amt[i] >= T2_MIN_USD and same[i] >= T2_COUNT,
               "T5": bool(par[i]) and known_before(par[i], a, w[i], False),
               "T5n": bool(par[i]) and known_before(par[i], a, w[i], True)}
        for r, h in hit.items():
            if h:
                fires.append((a, r, w[i]))
fires = pd.DataFrame(fires, columns=["ACID", "rule", "when"])
SETS = {"T1": ["T1"], "T3": ["T3"], "T5": ["T5"], "T5 narrow": ["T5n"], "T2": ["T2"],
        "RECOMMENDED: T1 or T3 or T5 narrow": ["T1", "T3", "T5n"], "RECOMMENDED + T2": ["T1", "T3", "T5n", "T2"]}
LABEL = {"T1": f"T1 account <= {T1_AGE_DAYS} days old, >= ${T1_USD/1e3:,.0f}k by IT in {T1_WINDOW}",
         "T3": f"T3 account <= {T3_AGE_DAYS} days old, >= {T3_SENDERS} distinct senders in {T3_WINDOW_DAYS} days",
         "T5": "T5 sender had already funded another mule", "T5 narrow": f"T5 narrow (ignores senders who paid > {BROAD_SENDER_MIN} other accounts)",
         "T2": f"T2 {T2_COUNT}rd identical inward TT today from one sender (each >= ${T2_MIN_USD/1e3:,.0f}k)"}

def alerts_after_cooldown(f):
    out = []
    for a, g in f.sort_values("when").groupby("ACID"):
        last = None
        for t in g.when:
            if last is None or t - last > pd.Timedelta(days=COOLDOWN_DAYS):
                out.append((a, t)); last = t
    return pd.DataFrame(out, columns=["ACID", "when"])

# ---------- (A1) what each mule would have looked like at its first alert ----------
mt_all = mule_tx.sort_values("when")
def at_alert(a, t):
    m = mt_all[mt_all.ACID == a]; before = m.when <= t
    ins, outs = m[m.flow == "in"], m[m.flow == "out"]
    bal = (np.where(m.is_cr, m.TRAN_AMT, -m.TRAN_AMT) * before).sum()
    out_tot = outs.TRAN_AMT.sum()
    return {"first alert": t, "days since opening": (t - m.ACCT_OPN_DATE.min()).days,
            "IT before alert $k": ins[ins.when <= t].TRAN_AMT.sum() / 1e3, "IT after $k": ins[ins.when > t].TRAN_AMT.sum() / 1e3,
            "out before alert $k": outs[outs.when <= t].TRAN_AMT.sum() / 1e3, "out after $k": outs[outs.when > t].TRAN_AMT.sum() / 1e3,
            "balance at alert $k": bal / 1e3,
            "share of outflow still to come": outs[outs.when > t].TRAN_AMT.sum() / out_tot if out_tot else np.nan}

rec = SETS["RECOMMENDED: T1 or T3 or T5 narrow"]
rows = []
for a in new_acids + old_acids:
    f = fires[(fires.ACID == a) & fires.rule.isin(rec)]
    r = {"account": str(names[a])[:26], "group": group[a],
         **{k: (fires[(fires.ACID == a) & (fires.rule == k)].when.min()) for k in ["T1", "T3", "T5n", "T2"]}}
    r.update(at_alert(a, f.when.min()) if len(f) else {"first alert": pd.NaT})
    rows.append(r)
M = pd.DataFrame(rows).set_index("account")
print("(A1) EACH MULE UNDER THE RECOMMENDED SET (T1 or T3 or T5 narrow), replayed in real time")
display(M.apply(lambda c: c.round(2) if c.dtype.kind == 'f' else c))
caught = M["first alert"].notna()
print(f"Caught: {int(caught[M.group == 'new mule'].sum())} of {int((M.group == 'new mule').sum())} new, "
      f"{int(caught[M.group == 'original mule'].sum())} of {int((M.group == 'original mule').sum())} original. "
      f"Median share of each caught mule's outflow still to come at the first alert: {M.loc[caught, 'share of outflow still to come'].median():.0%}")

# ---------- (A2) alert volume on everyone else, real-time, with cooldown ----------
oth_f = fires[~fires.ACID.isin(mule_set) & (fires.when >= TT_FROM)]
vol = []
for name, rules in SETS.items():
    al = alerts_after_cooldown(oth_f[oth_f.rule.isin(rules)])
    mf = fires[fires.ACID.isin(mule_set) & fires.rule.isin(rules)]
    vol.append({"rule set": LABEL.get(name, name),
                "new mules": int(mf.ACID.isin(new_acids).groupby(mf.ACID).any().sum()) if len(mf) else 0,
                "original mules": int(mf[mf.ACID.isin(old_acids)].ACID.nunique()),
                "alerts per month": round(len(al) / months, 1), "accounts alerted": al.ACID.nunique()})
volume_rt = pd.DataFrame(vol).set_index("rule set")
print(f"\n(A2) REAL-TIME ALERT VOLUME on non-mule accounts since {TT_FROM} ({months} complete months; "
      f"repeat alerts within {COOLDOWN_DAYS} days counted once)")
display(volume_rt)

# ---------- (A3) follow-up: outgoing payments from alerted accounts go to review before release ----------
al_rec = alerts_after_cooldown(oth_f[oth_f.rule.isin(rec)])
outs_o = ev[(ev.flow == "out") & ~ev.ACID.isin(mule_set) & (ev.when >= TT_FROM)][["ACID", "when", "TRAN_AMT"]]
held = outs_o.merge(al_rec.rename(columns={"when": "alert"}), on="ACID")
held = held[(held.when >= held.alert) & (held.when <= held.alert + pd.Timedelta(days=HOLD_DAYS))].drop_duplicates(["ACID", "when", "TRAN_AMT"])
print(f"\n(A3) HOLD-ON-OUTGOING: payments out of an alerted non-mule account within {HOLD_DAYS} days of its alert")
print(f"   {len(held) / months:.1f} payments per month for review (${held.TRAN_AMT.sum() / months / 1e3:,.0f}k per month), "
      f"on {held.ACID.nunique()} accounts over the period")
mh = []
for a in [x for x in new_acids + old_acids if pd.notna(M.loc[str(names[x])[:26], "first alert"])]:
    t = M.loc[str(names[a])[:26], "first alert"]
    o = mt_all[(mt_all.ACID == a) & (mt_all.flow == "out")]
    mh.append({"account": str(names[a])[:26], "would be held $k (next 7 days)":
               o[(o.when >= t) & (o.when <= t + pd.Timedelta(days=HOLD_DAYS))].TRAN_AMT.sum() / 1e3,
               "all outflow after alert $k": o[o.when > t].TRAN_AMT.sum() / 1e3})
if mh:
    print("   For the mules it catches:")
    display(pd.DataFrame(mh).set_index("account").round(0))

# ---------- (B) Review list: accounts paid by senders who also funded mules ----------
t1s = set(fires[fires.rule == "T1"].ACID); t3s = set(fires[fires.rule == "T3"].ACID)
ids = list(cand.index)
R = pd.DataFrame()
if ids:
    if SYNTHETIC:
        mst = syn_all[syn_all.ACID.isin(ids)].drop_duplicates("ACID")[["ACID", "FORACID", "ACCT_NAME", "ACCT_OPN_DATE"]]
    else:
        mst = read_chunks(f"SELECT ACID, FORACID, ACCT_NAME, CIF_ID, ACCT_OPN_DATE, ACCT_CLS_FLG, SCHM_CODE FROM {GAM} "
                          "WHERE ACID IN ({ids})", ids, fin())
    R = mst.set_index("ACID")
    ci = itc[itc.ACID.isin(ids)]
    ring = set(shared.index) - broad
    R["ring senders"] = [", ".join(sorted(set(ci[ci.ACID == a].party) & ring))[:70] for a in R.index]
    R["n ring senders"] = [len(set(ci[ci.ACID == a].party) & ring) for a in R.index]
    R["n broad senders"] = [len(set(ci[ci.ACID == a].party) & broad) for a in R.index]
    R["from shared senders $k"] = cand.received_k.reindex(R.index)
    bf = book_feat.reindex(R.index)
    R["all IT $k"] = bf["IT total $k"].round(0); R["all senders"] = bf["distinct senders"]
    R["out / IT"] = bf["out / IT"].round(2); R["out within 72h"] = bf["IT money out within 72h"].round(2)
    R["days open at first IT"] = bf["days open at first IT"]
    R["T1"] = R.index.isin(list(t1s)); R["T3"] = R.index.isin(list(t3s))
    try:
        sm = pd.read_csv("outputs/early_signals_scan/scan_metrics.csv", usecols=["ACID", "first_hit_day"])
        R["awakening rule"] = R.index.map(sm.set_index("ACID").first_hit_day.notna()).fillna(False)
    except Exception:
        R["awakening rule"] = "n/a"
    try:
        nr = pd.read_csv("outputs/network_onboarding/accounts_in_reach.csv", usecols=["ACID", "hop"])
        R["linked to mule owners"] = R.index.map(nr.drop_duplicates("ACID").set_index("ACID").hop).fillna("")
    except Exception:
        R["linked to mule owners"] = "n/a"
    try:
        if SYNTHETIC:
            raise RuntimeError("synthetic")
        leads = pd.read_sql("SELECT * FROM aml_top_leads", rep()); leads.columns = [c.lower() for c in leads.columns]
        R["production lead"] = R.index.isin(set(leads["acid"].astype(str)))
    except Exception:
        R["production lead"] = "n/a"
    R["tier"] = np.select([R["n ring senders"] >= 2, (R["n ring senders"] == 1) & (R["from shared senders $k"] >= 10)],
                          ["A: 2+ ring senders", "B: 1 ring sender, >= $10k"], "C: broad senders or tiny amounts")
    R = R.sort_values(["tier", "from shared senders $k"], ascending=[True, False])
    print(f"\n(B) REVIEW LIST: {len(R)} accounts paid by senders who also funded mules — linked by who paid them, not confirmed mules")
    display(R)

M.to_csv(OUT / "rt_mules_at_first_alert.csv"); volume_rt.to_csv(OUT / "rt_alert_volume.csv")
if len(R):
    R.to_csv(OUT / "tt_review_list.csv")
print("Saved rt_mules_at_first_alert.csv, rt_alert_volume.csv" + (", tt_review_list.csv" if len(R) else ""), "to", OUT.resolve())

(A1) EACH MULE UNDER THE RECOMMENDED SET (T1 or T3 or T5 narrow), replayed in real time


,group,T1,T3,T5n,T2,first alert,days since opening,IT before alert $k,IT after $k,out before alert $k,out after $k,balance at alert $k,share of outflow still to come
account,,,,,,,,,,,,,
TYR TECH -SMC LTD,new mule,2026-03-12 08:00:55,2026-03-16 18:31:24,NaT,NaT,2026-03-12 08:00:55,14.0,575.50,4498.70,462.85,4609.16,111.94,0.91
THRIFTOPIA UGANDA LIMITED,new mule,2026-04-01 08:11:41,NaT,2026-04-01 08:11:41,2026-04-01 08:11:42,2026-04-01 08:11:41,37.0,1144.90,1144.90,0.00,2289.60,1144.90,1.00
QUENSPIRE UGANDA LIMITED,new mule,2026-04-15 08:11:38,NaT,2026-04-10 08:11:38,NaT,2026-04-10 08:11:38,3.0,473.80,262.76,0.08,736.24,473.81,1.00
PACIFIC PRINCE INTERNATION,original mule,NaT,NaT,2025-03-28 08:02:38,NaT,2025-03-28 08:02:38,189.0,6124.77,5407.10,6084.91,5437.83,34.76,0.47
HESHIMA PRIME LOGISTICS LT,original mule,NaT,NaT,NaT,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AETHERGRID TECHNOLOGIES LI,original mule,2025-03-20 18:34:03,2025-03-20 08:00:33,2025-02-26 08:00:59,NaT,2025-02-26 08:00:59,6.0,37.00,5962.60,0.00,5994.61,37.16,1.00
EMIC VENTURES,original mule,NaT,NaT,NaT,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
NEBULACORE INNOVATIONS LIM,original mule,2025-03-12 08:01:11,2025-03-11 13:13:32,2025-02-24 08:12:43,NaT,2025-02-24 08:12:43,4.0,25.00,4333.40,0.00,4378.70,25.08,1.00
KWELIVEST INVESTMENT SMC L,original mule,2025-05-05 18:05:24,2025-05-02 08:04:40,2025-04-22 08:03:16,NaT,2025-04-22 08:03:16,61.0,10.44,2787.55,0.01,2807.62,10.57,1.00


Caught: 3 of 3 new, 6 of 10 original. Median share of each caught mule's outflow still to come at the first alert: 100%

(A2) REAL-TIME ALERT VOLUME on non-mule accounts since 2025-01-01 (21 complete months; repeat alerts within 30 days counted once)


,new mules,original mules,alerts per month,accounts alerted
rule set,,,,
"T1 account <= 90 days old, >= $500k by IT in 7D",3,5,4.3,72
"T3 account <= 90 days old, >= 20 distinct senders in 30 days",1,3,1.6,23
T5 sender had already funded another mule,3,5,6.4,76
T5 narrow (ignores senders who paid > 10 other accounts),2,5,5.9,71
T2 3rd identical inward TT today from one sender (each >= $10k),1,0,1.3,20
RECOMMENDED: T1 or T3 or T5 narrow,3,6,9.6,127
RECOMMENDED + T2,3,6,10.9,147



(A3) HOLD-ON-OUTGOING: payments out of an alerted non-mule account within 7 days of its alert
   283.5 payments per month for review ($11,909k per month), on 121 accounts over the period
   For the mules it catches:


,would be held $k (next 7 days),all outflow after alert $k
account,,
TYR TECH -SMC LTD,2362.0,4609.0
THRIFTOPIA UGANDA LIMITED,2290.0,2290.0
QUENSPIRE UGANDA LIMITED,599.0,736.0
PACIFIC PRINCE INTERNATION,713.0,5438.0
AETHERGRID TECHNOLOGIES LI,47.0,5995.0
NEBULACORE INNOVATIONS LIM,0.0,4379.0
KWELIVEST INVESTMENT SMC L,94.0,2808.0
OMOSPIRE UGANDA LIMITED,108.0,108.0
RAVO TECH COMPANY SMC LIMI,1108.0,2007.0



(B) REVIEW LIST: 44 accounts paid by senders who also funded mules — linked by who paid them, not confirmed mules


,FORACID,ACCT_NAME,CIF_ID,ACCT_OPN_DATE,ACCT_CLS_FLG,SCHM_CODE,ring senders,n ring senders,n broad senders,from shared senders $k,all IT $k,all senders,out / IT,out within 72h,days open at first IT,T1,T3,awakening rule,linked to mule owners,production lead,tier
ACID,,,,,,,,,,,,,,,,,,,,,
UG5856411,9030023403071,AFRASIA PARTNERS CO. LIMITED,20301171115,2023-08-29,Y,OD005,"JAMES NADEL, KHAM HOANG DO, KIM CHI THI NGUYEN, LIEN N PHAM, LOAN ...",7,1,1403,5688.0,65,1.00,0.77,492,False,False,False,2: shares a party,n/a,A: 2+ ring senders
UG6160539,9030026394510,NEURONOVA COMPANY - SMC LIMITED,20301183299,2025-05-19,N,OD005,"DUYKIM TRAN, JIMMY HOANG, JOHN LASPINA, KRISTY P DINH, TIFFANY CAO",5,1,981,3765.0,43,1.00,0.97,4,True,True,True,,n/a,A: 2+ ring senders
UG6103353,9030025819887,ZURIFIN TECHNOLOGY CO SMC LIMITED,20301181115,2025-01-22,N,OD005,"JAMES NADEL, KHAM HOANG DO, KIM CHI THI NGUYEN, LIEN N PHAM, QUYEN...",7,1,212,1516.0,55,1.00,0.88,12,True,True,True,,n/a,A: 2+ ring senders
UG6161933,9030026409240,ENZO TECH SYSTEMS UGANDA - SMC LIMITED,20301183360,2025-05-22,Y,OD005,"LOAN L BUI, MY HA THI TRAN",2,0,103,1048.0,18,1.00,0.57,25,False,False,True,,n/a,A: 2+ ring senders
UG6159107,9030026380889,ASTITUTE ROYAL INVESTMENTS LTD,20301183255,2025-05-16,Y,OD005,"CASSIDY NGUYEN CHI, VINH DUC NGUYEN",2,0,95,3499.0,37,1.01,0.77,31,True,True,True,,n/a,A: 2+ ring senders
UG344638,031075USD5400852,UGX USD POSITION SUSPENSE,None,1900-01-01,N,TRADE,"ALLEN FRENCH, AMY WOLFE, ANITA TRINH, BENNY MORITZ, BILLY NGO, BRE...",47,3,7,85.0,748,1848.54,1.00,45657,False,False,False,,n/a,A: 2+ ring senders
UG351989,031006USD5400852,UGX USD POSITION SUSPENSE,None,1900-01-01,N,TRADE,"BENNY MORITZ, BRET BISHOP, CHI LUU LLC DBA LIVING COLORS NAILS, CH...",30,3,4,798.0,3972,565.49,1.00,45657,False,False,False,,n/a,A: 2+ ring senders
UG192626,031003USD5400852,UGX USD POSITION SUSPENSE,None,1900-01-01,N,TRADE,"DUYKIM TRAN, JAMES NADEL, JOHN LASPINA, KHAM HOANG DO, KIM CHI THI...",12,2,1,49.0,602,653.38,1.00,45657,False,False,False,,n/a,A: 2+ ring senders
UG6160591,9030026394545,NEON PULSE INNOVATIONS COMPANY SMC LIMITED,20301183298,2025-05-19,N,OD005,THANG PHAM,1,0,775,4909.0,23,1.00,0.97,14,True,False,True,,n/a,"B: 1 ring sender, >= $10k"


Saved rt_mules_at_first_alert.csv, rt_alert_volume.csv, tt_review_list.csv to /home/A270827/experiments/AML/outputs/tt_indicators


In [15]:
# Run inside tt_indicators.ipynb after all its sections:   %run -i tt_candidates_and_ruleset.py
# (A) REAL-TIME REPLAY of the candidate rules, as a near-real-time system would run them: transactions are replayed in
#     time order and each rule fires on an inward TT using only what was known at that moment.
# (B) The review list of accounts paid by senders who also funded mules, enriched.
# No new transaction pulls (one small account-master query for the review list).

T1_AGE_DAYS, T1_USD, T1_WINDOW = 90, 500_000, "7D"        # young account, large inflow by inward TT
T3_AGE_DAYS, T3_SENDERS, T3_WINDOW_DAYS = 90, 20, 30      # young account, many distinct senders
T2_COUNT, T2_MIN_USD = 3, 10_000                          # identical inward TTs, same sender, same day
BROAD_SENDER_MIN = 10                                     # a mule-linked sender who paid > this many other accounts is "broad"
COOLDOWN_DAYS = 30                                        # repeat alerts of the same rule on one account within this are one alert
HOLD_DAYS = 7                                             # outgoing payments within this many days after an alert go to review

broad = set(shared[shared.other_accounts > BROAD_SENDER_MIN].index)
ev = pd.concat([mule_tx[mule_tx.flow != ""], others[others.flow != ""]], ignore_index=True)
ev = ev[["ACID", "when", "TRAN_DATE", "TRAN_AMT", "flow", "party", "ACCT_OPN_DATE"]].dropna(subset=["when"])
ev = ev.sort_values(["ACID", "when"]).reset_index(drop=True)
mule_set = set(mule_acids)

# when did each sender first fund each mule (for "known sender": funded ANOTHER mule BEFORE this moment)
mi_ = ev[ev.ACID.isin(mule_set) & (ev.flow == "in") & (ev.party.str.len() > 3)]
first_fund = mi_.groupby(["party", "ACID"]).when.min()
fund_by_party = {p: g.droplevel(0).sort_values() for p, g in first_fund.groupby(level=0)}

def known_before(party, acct, t, exclude_broad):
    s = fund_by_party.get(party)
    if s is None or (exclude_broad and party in broad):
        return False
    s = s.drop(acct, errors="ignore")
    return bool(len(s)) and s.iloc[0] < t

# ---------- replay every inward TT ----------
fires = []
d7, d30 = pd.Timedelta(T1_WINDOW), pd.Timedelta(days=T3_WINDOW_DAYS)
for a, g in ev[ev.flow == "in"].groupby("ACID", sort=False):
    w = g.when.to_numpy(); amt = g.TRAN_AMT.to_numpy(); par = g.party.to_numpy()
    age = (g.when - g.ACCT_OPN_DATE).dt.days.to_numpy()
    same = (g.assign(big=g.TRAN_AMT >= T2_MIN_USD).groupby([g.TRAN_DATE, g.party, g.TRAN_AMT]).cumcount() + 1).to_numpy()
    lo7 = lo30 = 0; s7 = 0.0; cnt = {}
    for i in range(len(g)):
        s7 += amt[i]
        while w[lo7] <= w[i] - d7:
            s7 -= amt[lo7]; lo7 += 1
        if par[i]:
            cnt[par[i]] = cnt.get(par[i], 0) + 1
        while w[lo30] <= w[i] - d30:
            if par[lo30]:
                cnt[par[lo30]] -= 1
                if not cnt[par[lo30]]:
                    del cnt[par[lo30]]
            lo30 += 1
        hit = {"T1": age[i] <= T1_AGE_DAYS and s7 >= T1_USD,
               "T3": age[i] <= T3_AGE_DAYS and len(cnt) >= T3_SENDERS,
               "T2": amt[i] >= T2_MIN_USD and same[i] >= T2_COUNT,
               "T5": bool(par[i]) and known_before(par[i], a, w[i], False),
               "T5n": bool(par[i]) and known_before(par[i], a, w[i], True)}
        for r, h in hit.items():
            if h:
                fires.append((a, r, w[i]))
fires = pd.DataFrame(fires, columns=["ACID", "rule", "when"])
SETS = {"T1": ["T1"], "T3": ["T3"], "T5": ["T5"], "T5 narrow": ["T5n"], "T2": ["T2"],
        "RECOMMENDED: T1 or T3 or T5 narrow": ["T1", "T3", "T5n"], "RECOMMENDED + T2": ["T1", "T3", "T5n", "T2"]}
LABEL = {"T1": f"T1 account <= {T1_AGE_DAYS} days old, >= ${T1_USD/1e3:,.0f}k by IT in {T1_WINDOW}",
         "T3": f"T3 account <= {T3_AGE_DAYS} days old, >= {T3_SENDERS} distinct senders in {T3_WINDOW_DAYS} days",
         "T5": "T5 sender had already funded another mule", "T5 narrow": f"T5 narrow (ignores senders who paid > {BROAD_SENDER_MIN} other accounts)",
         "T2": f"T2 {T2_COUNT}rd identical inward TT today from one sender (each >= ${T2_MIN_USD/1e3:,.0f}k)"}

def alerts_after_cooldown(f):
    out = []
    for a, g in f.sort_values("when").groupby("ACID"):
        last = None
        for t in g.when:
            if last is None or t - last > pd.Timedelta(days=COOLDOWN_DAYS):
                out.append((a, t)); last = t
    return pd.DataFrame(out, columns=["ACID", "when"])

# ---------- (A1) what each mule would have looked like at its first alert ----------
mt_all = mule_tx.sort_values("when")
def at_alert(a, t):
    m = mt_all[mt_all.ACID == a]; before = m.when <= t
    ins, outs = m[m.flow == "in"], m[m.flow == "out"]
    bal = (np.where(m.is_cr, m.TRAN_AMT, -m.TRAN_AMT) * before).sum()
    out_tot = outs.TRAN_AMT.sum()
    return {"first alert": t, "days since opening": (t - m.ACCT_OPN_DATE.min()).days,
            "IT before alert $k": ins[ins.when <= t].TRAN_AMT.sum() / 1e3, "IT after $k": ins[ins.when > t].TRAN_AMT.sum() / 1e3,
            "out before alert $k": outs[outs.when <= t].TRAN_AMT.sum() / 1e3, "out after $k": outs[outs.when > t].TRAN_AMT.sum() / 1e3,
            "balance at alert $k": bal / 1e3,
            "share of outflow still to come": outs[outs.when > t].TRAN_AMT.sum() / out_tot if out_tot else np.nan}

rec = SETS["RECOMMENDED: T1 or T3 or T5 narrow"]
rows = []
for a in new_acids + old_acids:
    f = fires[(fires.ACID == a) & fires.rule.isin(rec)]
    r = {"account": str(names[a])[:26], "group": group[a],
         **{k: (fires[(fires.ACID == a) & (fires.rule == k)].when.min()) for k in ["T1", "T3", "T5n", "T2"]}}
    r.update(at_alert(a, f.when.min()) if len(f) else {"first alert": pd.NaT})
    rows.append(r)
M = pd.DataFrame(rows).set_index("account")
print("(A1) EACH MULE UNDER THE RECOMMENDED SET (T1 or T3 or T5 narrow), replayed in real time")
display(M.apply(lambda c: c.round(2) if c.dtype.kind == 'f' else c))
caught = M["first alert"].notna()
print(f"Caught: {int(caught[M.group == 'new mule'].sum())} of {int((M.group == 'new mule').sum())} new, "
      f"{int(caught[M.group == 'original mule'].sum())} of {int((M.group == 'original mule').sum())} original. "
      f"Median share of each caught mule's outflow still to come at the first alert: {M.loc[caught, 'share of outflow still to come'].median():.0%}")

# ---------- (A2) alert volume on everyone else, real-time, with cooldown ----------
oth_f = fires[~fires.ACID.isin(mule_set) & (fires.when >= TT_FROM)]
vol = []
for name, rules in SETS.items():
    al = alerts_after_cooldown(oth_f[oth_f.rule.isin(rules)])
    mf = fires[fires.ACID.isin(mule_set) & fires.rule.isin(rules)]
    vol.append({"rule set": LABEL.get(name, name),
                "new mules": int(mf.ACID.isin(new_acids).groupby(mf.ACID).any().sum()) if len(mf) else 0,
                "original mules": int(mf[mf.ACID.isin(old_acids)].ACID.nunique()),
                "alerts per month": round(len(al) / months, 1), "accounts alerted": al.ACID.nunique()})
volume_rt = pd.DataFrame(vol).set_index("rule set")
print(f"\n(A2) REAL-TIME ALERT VOLUME on non-mule accounts since {TT_FROM} ({months} complete months; "
      f"repeat alerts within {COOLDOWN_DAYS} days counted once)")
display(volume_rt)

# ---------- (A3) follow-up: outgoing payments from alerted accounts go to review before release ----------
al_rec = alerts_after_cooldown(oth_f[oth_f.rule.isin(rec)])
outs_o = ev[(ev.flow == "out") & ~ev.ACID.isin(mule_set) & (ev.when >= TT_FROM)][["ACID", "when", "TRAN_AMT"]]
held = outs_o.merge(al_rec.rename(columns={"when": "alert"}), on="ACID")
held = held[(held.when >= held.alert) & (held.when <= held.alert + pd.Timedelta(days=HOLD_DAYS))].drop_duplicates(["ACID", "when", "TRAN_AMT"])
print(f"\n(A3) HOLD-ON-OUTGOING: payments out of an alerted non-mule account within {HOLD_DAYS} days of its alert")
print(f"   {len(held) / months:.1f} payments per month for review (${held.TRAN_AMT.sum() / months / 1e3:,.0f}k per month), "
      f"on {held.ACID.nunique()} accounts over the period")
mh = []
for a in [x for x in new_acids + old_acids if pd.notna(M.loc[str(names[x])[:26], "first alert"])]:
    t = M.loc[str(names[a])[:26], "first alert"]
    o = mt_all[(mt_all.ACID == a) & (mt_all.flow == "out")]
    mh.append({"account": str(names[a])[:26], "would be held $k (next 7 days)":
               o[(o.when >= t) & (o.when <= t + pd.Timedelta(days=HOLD_DAYS))].TRAN_AMT.sum() / 1e3,
               "all outflow after alert $k": o[o.when > t].TRAN_AMT.sum() / 1e3})
if mh:
    print("   For the mules it catches:")
    display(pd.DataFrame(mh).set_index("account").round(0))

# ---------- (A4) calibrating the follow-up: which alerts, how long to watch, which payments ----------
# After an alert the account is "on watch" for W days; outgoing payments of at least $P during the watch go to review.
mule_out = mt_all[mt_all.flow == "out"][["ACID", "when", "TRAN_AMT"]]
mule_out_total = mule_out[mule_out.ACID.isin(set(fires[fires.ACID.isin(mule_set) & fires.rule.isin(rec)].ACID))].TRAN_AMT.sum()
cal = []
for src_name, src in (("any recommended rule", rec), ("young-account rules only (T1, T3)", ["T1", "T3"])):
    al_o = alerts_after_cooldown(oth_f[oth_f.rule.isin(src)]).rename(columns={"when": "alert"})
    mo = oth_m = fires[fires.ACID.isin(mule_set) & fires.rule.isin(src)].groupby("ACID").when.min().rename("alert").reset_index()
    jo, jm = outs_o.merge(al_o, on="ACID"), mule_out.merge(mo, on="ACID")
    for W in (7, 30, 90):
        for P in (0, 50_000, 250_000):
            ho = jo[(jo.when >= jo.alert) & (jo.when <= jo.alert + pd.Timedelta(days=W)) & (jo.TRAN_AMT >= P)].drop_duplicates(["ACID", "when", "TRAN_AMT"])
            hm = jm[(jm.when >= jm.alert) & (jm.when <= jm.alert + pd.Timedelta(days=W)) & (jm.TRAN_AMT >= P)]
            cal.append({"alerts from": src_name, "watch days": W, "payments >= $": f"{P:,}",
                        "reviews per month": round(len(ho) / months, 1), "$k per month reviewed": round(ho.TRAN_AMT.sum() / months / 1e3),
                        "mules covered": hm.ACID.nunique(), "mule outflow held $k": round(hm.TRAN_AMT.sum() / 1e3),
                        "share of caught mules' outflow held": round(hm.TRAN_AMT.sum() / mule_out_total, 2) if mule_out_total else np.nan})
calib = pd.DataFrame(cal).set_index(["alerts from", "watch days", "payments >= $"])
print("\n(A4) CALIBRATING THE FOLLOW-UP: reviews per month on other accounts vs mule money that would have been held")
display(calib)
calib.to_csv(OUT / "rt_hold_calibration.csv")

# ---------- (B) Review list: accounts paid by senders who also funded mules ----------
t1s = set(fires[fires.rule == "T1"].ACID); t3s = set(fires[fires.rule == "T3"].ACID)
ids = list(cand.index)
R = pd.DataFrame()
if ids:
    if SYNTHETIC:
        mst = syn_all[syn_all.ACID.isin(ids)].drop_duplicates("ACID")[["ACID", "FORACID", "ACCT_NAME", "ACCT_OPN_DATE"]]
    else:
        mst = read_chunks(f"SELECT ACID, FORACID, ACCT_NAME, CIF_ID, ACCT_OPN_DATE, ACCT_CLS_FLG, SCHM_CODE FROM {GAM} "
                          "WHERE ACID IN ({ids})", ids, fin())
    R = mst.set_index("ACID")
    ci = itc[itc.ACID.isin(ids)]
    ring = set(shared.index) - broad
    R["ring senders"] = [", ".join(sorted(set(ci[ci.ACID == a].party) & ring))[:70] for a in R.index]
    R["n ring senders"] = [len(set(ci[ci.ACID == a].party) & ring) for a in R.index]
    R["n broad senders"] = [len(set(ci[ci.ACID == a].party) & broad) for a in R.index]
    R["from shared senders $k"] = cand.received_k.reindex(R.index)
    bf = book_feat.reindex(R.index)
    R["all IT $k"] = bf["IT total $k"].round(0); R["all senders"] = bf["distinct senders"]
    R["out / IT"] = bf["out / IT"].round(2); R["out within 72h"] = bf["IT money out within 72h"].round(2)
    R["days open at first IT"] = bf["days open at first IT"]
    R["T1"] = R.index.isin(list(t1s)); R["T3"] = R.index.isin(list(t3s))
    try:
        sm = pd.read_csv("outputs/early_signals_scan/scan_metrics.csv", usecols=["ACID", "first_hit_day"])
        R["awakening rule"] = R.index.map(sm.set_index("ACID").first_hit_day.notna()).fillna(False)
    except Exception:
        R["awakening rule"] = "n/a"
    try:
        nr = pd.read_csv("outputs/network_onboarding/accounts_in_reach.csv", usecols=["ACID", "hop"])
        R["linked to mule owners"] = R.index.map(nr.drop_duplicates("ACID").set_index("ACID").hop).fillna("")
    except Exception:
        R["linked to mule owners"] = "n/a"
    try:
        if SYNTHETIC:
            raise RuntimeError("synthetic")
        leads = pd.read_sql("SELECT * FROM aml_top_leads", rep()); leads.columns = [c.lower() for c in leads.columns]
        R["production lead"] = R.index.isin(set(leads["acid"].astype(str)))
    except Exception:
        R["production lead"] = "n/a"
    R["tier"] = np.select([R["n ring senders"] >= 2, (R["n ring senders"] == 1) & (R["from shared senders $k"] >= 10)],
                          ["A: 2+ ring senders", "B: 1 ring sender, >= $10k"], "C: broad senders or tiny amounts")
    R = R.sort_values(["tier", "from shared senders $k"], ascending=[True, False])
    print(f"\n(B) REVIEW LIST: {len(R)} accounts paid by senders who also funded mules — linked by who paid them, not confirmed mules")
    display(R)

M.to_csv(OUT / "rt_mules_at_first_alert.csv"); volume_rt.to_csv(OUT / "rt_alert_volume.csv")
if len(R):
    R.to_csv(OUT / "tt_review_list.csv")
print("Saved rt_mules_at_first_alert.csv, rt_alert_volume.csv" + (", tt_review_list.csv" if len(R) else ""), "to", OUT.resolve())

(A1) EACH MULE UNDER THE RECOMMENDED SET (T1 or T3 or T5 narrow), replayed in real time


,group,T1,T3,T5n,T2,first alert,days since opening,IT before alert $k,IT after $k,out before alert $k,out after $k,balance at alert $k,share of outflow still to come
account,,,,,,,,,,,,,
TYR TECH -SMC LTD,new mule,2026-03-12 08:00:55,2026-03-16 18:31:24,NaT,NaT,2026-03-12 08:00:55,14.0,575.50,4498.70,462.85,4609.16,111.94,0.91
THRIFTOPIA UGANDA LIMITED,new mule,2026-04-01 08:11:41,NaT,2026-04-01 08:11:41,2026-04-01 08:11:42,2026-04-01 08:11:41,37.0,1144.90,1144.90,0.00,2289.60,1144.90,1.00
QUENSPIRE UGANDA LIMITED,new mule,2026-04-15 08:11:38,NaT,2026-04-10 08:11:38,NaT,2026-04-10 08:11:38,3.0,473.80,262.76,0.08,736.24,473.81,1.00
PACIFIC PRINCE INTERNATION,original mule,NaT,NaT,2025-03-28 08:02:38,NaT,2025-03-28 08:02:38,189.0,6124.77,5407.10,6084.91,5437.83,34.76,0.47
HESHIMA PRIME LOGISTICS LT,original mule,NaT,NaT,NaT,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AETHERGRID TECHNOLOGIES LI,original mule,2025-03-20 18:34:03,2025-03-20 08:00:33,2025-02-26 08:00:59,NaT,2025-02-26 08:00:59,6.0,37.00,5962.60,0.00,5994.61,37.16,1.00
EMIC VENTURES,original mule,NaT,NaT,NaT,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
NEBULACORE INNOVATIONS LIM,original mule,2025-03-12 08:01:11,2025-03-11 13:13:32,2025-02-24 08:12:43,NaT,2025-02-24 08:12:43,4.0,25.00,4333.40,0.00,4378.70,25.08,1.00
KWELIVEST INVESTMENT SMC L,original mule,2025-05-05 18:05:24,2025-05-02 08:04:40,2025-04-22 08:03:16,NaT,2025-04-22 08:03:16,61.0,10.44,2787.55,0.01,2807.62,10.57,1.00


Caught: 3 of 3 new, 6 of 10 original. Median share of each caught mule's outflow still to come at the first alert: 100%

(A2) REAL-TIME ALERT VOLUME on non-mule accounts since 2025-01-01 (21 complete months; repeat alerts within 30 days counted once)


,new mules,original mules,alerts per month,accounts alerted
rule set,,,,
"T1 account <= 90 days old, >= $500k by IT in 7D",3,5,4.3,72
"T3 account <= 90 days old, >= 20 distinct senders in 30 days",1,3,1.6,23
T5 sender had already funded another mule,3,5,6.4,76
T5 narrow (ignores senders who paid > 10 other accounts),2,5,5.9,71
T2 3rd identical inward TT today from one sender (each >= $10k),1,0,1.3,20
RECOMMENDED: T1 or T3 or T5 narrow,3,6,9.6,127
RECOMMENDED + T2,3,6,10.9,147



(A3) HOLD-ON-OUTGOING: payments out of an alerted non-mule account within 7 days of its alert
   283.5 payments per month for review ($11,909k per month), on 121 accounts over the period
   For the mules it catches:


,would be held $k (next 7 days),all outflow after alert $k
account,,
TYR TECH -SMC LTD,2362.0,4609.0
THRIFTOPIA UGANDA LIMITED,2290.0,2290.0
QUENSPIRE UGANDA LIMITED,599.0,736.0
PACIFIC PRINCE INTERNATION,713.0,5438.0
AETHERGRID TECHNOLOGIES LI,47.0,5995.0
NEBULACORE INNOVATIONS LIM,0.0,4379.0
KWELIVEST INVESTMENT SMC L,94.0,2808.0
OMOSPIRE UGANDA LIMITED,108.0,108.0
RAVO TECH COMPANY SMC LIMI,1108.0,2007.0



(A4) CALIBRATING THE FOLLOW-UP: reviews per month on other accounts vs mule money that would have been held


reviews per month  $k per month reviewed  mules covered  mule outflow held $k  share of caught mules' outflow held
alerts from                       watch days payments >= $                                                                                                                    
any recommended rule              7          0                          283.5                  11909              9                  7320                                 0.20
                                             50,000                      51.6                  10916              5                  6733                                 0.19
                                             250,000                     10.1                   6307              3                  2671                                 0.07
                                  30         0                         1111.6                  43490              9                 15671                                 0.44
                                             50,000                     183.5                  39815              8                 14086                                 0.39
                                             250,000                     38.5                  23489              6                  4069                                 0.11
                                  90         0                         2091.2                  92694              9                 27591                                 0.77
                                             50,000                     322.2                  86004              8                 24531                                 0.68
                                             250,000                     72.7                  58433              7                  6755                                 0.19
young-account rules only (T1, T3) 7          0                           33.3                   4912              8                  7690                                 0.21
                                             50,000                      18.0                   4665              7                  7081                                 0.20
                                             250,000                      3.5                   2975              4                  2912                                 0.08
                                  30         0                           99.9                  13200              8                 14613                                 0.41
                                             50,000                      49.8                  12460              7                 13230                                 0.37
                                             250,000                      8.8                   7679              6                  3822                                 0.11
                                  90         0                          159.8                  32448              8                 21126                                 0.59
                                             50,000                      67.0                  31184              7                 18698                                 0.52
                                             250,000                     12.8                  24986              6                  5341                                 0.15

PendingRollbackError: Can't reconnect until invalid transaction is rolled back.  Please rollback() fully before proceeding (Background on this error at: https://sqlalche.me/e/20/8s2b)